In [1]:
from get_stock_data import get_combined_tech_stocks
from get_normalized_price import get_normalized_prices
import numpy as np

In [2]:
tech_tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META"]
data = get_combined_tech_stocks(tech_tickers)

Fetching data from 2011-07-07 to 2026-07-03...


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


In [15]:
a = get_normalized_prices(data, "AAPL")

In [44]:
def get_history_and_target_price(target_ticker, window=50):
    target_price = get_normalized_prices(data, target_ticker)[:window]
    history_prices = {}
    for ticker in data.keys():
        history_prices[ticker] = get_normalized_prices(data, ticker)[window:]
    return target_price, history_prices

In [45]:
target_price, historical_prices = get_history_and_target_price("AAPL", 50)

In [ ]:
def compute_dtw_matrix(x, y, window=5):
    N, M = len(x), len(y)
    # Initialize with Infinity
    dtw_matrix = np.full((N + 1, M + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    # Sakoe-Chiba constraint band
    w = max(window, abs(N - M))
    
    for i in range(1, N + 1):
        # Only calculate within the diagonal band
        for j in range(max(1, i - w), min(M + 1, i + w + 1)):
            cost = abs(x[i - 1] - y[j - 1])
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i - 1, j],    # insertion
                dtw_matrix[i, j - 1],    # deletion
                dtw_matrix[i - 1, j - 1] # match
            )
    return dtw_matrix

def get_dtw_path_and_steps(dtw_matrix):
    i, j = dtw_matrix.shape[0] - 1, dtw_matrix.shape[1] - 1
    path = [(i - 1, j - 1)]
    
    while i > 1 or j > 1:
        neighbors = {}
        # Only check neighbors that are not infinity
        if dtw_matrix[i - 1, j - 1] != np.inf:
            neighbors["match"] = (i - 1, j - 1, dtw_matrix[i - 1, j - 1])
        if dtw_matrix[i - 1, j] != np.inf:
            neighbors["insert"] = (i - 1, j, dtw_matrix[i - 1, j])
        if dtw_matrix[i, j - 1] != np.inf:
            neighbors["delete"] = (i, j - 1, dtw_matrix[i, j - 1])
            
        best_move = min(neighbors.values(), key=lambda x: x[2])
        i, j = best_move[0], best_move[1]
        path.append((i - 1, j - 1))
        
    path.reverse()
    num_steps = len(path)
    
    return path, num_steps